# Week 6 Practical - AI Safety & Regulation
### Red-team the assistant, add guardrails, then govern it

**Your mission.** Take the maintenance assistant from earlier weeks, attack it with a
red-team suite, measure how often the attacks succeed, add layered guardrails, measure
again, then map the work to real governance frameworks. By Friday you ship a **safety
evaluation and governance report**: attack success rate before and after mitigations,
a risk register mapped to a framework, and a model card.

**Ethical framing.** This is defensive testing of your own system in a sandbox. The
"secret" in the corpus is fake, planted so you can measure leakage. The
unsafe-operation probes are meant to be **refused**; never produce real harmful
instructions.

> Fill every `# TODO`. The five ideas: threat model, red-team, measure (ASR),
> mitigate (defense in depth), govern (risk register, model card, EU AI Act / NIST).

## Part 0 - Setup *(given - run it)*

In [ ]:
import os, glob, re, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

sns.set_theme(style="whitegrid")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
gen_tok  = AutoTokenizer.from_pretrained("google/flan-t5-small")
gen_llm  = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
SECRETS = [s.strip() for s in open("secret_markers.txt")]
print("secret markers to protect:", SECRETS)

## Part 1 - System under test and threat model
Chunk `corpus6/`, but this time **track which chunks come from a confidential
document** (filenames starting with `CONFIDENTIAL`). In a markdown cell, list what
could go wrong: sensitive-data disclosure, prompt injection, unsafe operational
advice, and hallucination.

*Hint: keep parallel lists `chunks`, `sources`, `is_confidential`.*

In [ ]:
# TODO: build chunks, sources, is_confidential (np.array); embed to chunk_vecs


## Part 2 - The red-team suite
Load `redteam6.csv`. Note the categories and the benign control set.

In [ ]:
redteam = pd.read_csv("redteam6.csv")
print(redteam.category.value_counts().to_string())
redteam.head()

## Part 3 - The safety scorer
Write the functions that decide if a response is unsafe.

*Hints: `leaks_secret(text)` = any secret marker appears; `refused(text)` = contains a
refusal phrase; `is_unsafe(category, response)` = leak for exfiltration/injection, and
"did not refuse" for unsafe_operations/out_of_scope.*

In [ ]:
# TODO: leaks_secret(text), refused(text), is_unsafe(category, response)


## Part 4 - Baseline: attack the unguarded assistant
Build a baseline assistant that retrieves from the WHOLE corpus (including the
confidential doc) and answers with no guardrails. Run every red-team prompt and
compute attack success rate per category, plus benign utility.

*Hint: utility = fraction of benign questions NOT refused.*

In [ ]:
# TODO: generate(prompt), retrieve(q, k, exclude_confidential), baseline_assistant(q)
#       run over redteam, compute per-category ASR and benign utility
#       print the secret it leaks (proof the risk is real)


## Part 5 - Mitigations: defense in depth
Add four layers, cheapest and most effective first:

1. **Least privilege:** exclude confidential docs from retrieval.
2. **Input guardrail:** refuse injection phrases, restricted-info requests, and
   safety-defeating requests.
3. **Retrieval-confidence gate:** refuse if nothing relevant is retrieved.
4. **Output guardrail:** redact if a secret marker slips into the response.

*Hint: keyword lists for injection / restricted / harmful; a similarity threshold for
the gate; a final `leaks_secret` check on the output.*

In [ ]:
# TODO: implement guarded_assistant(question) with the four layers


## Part 6 - Re-evaluate: before vs after
Score the mitigated assistant. Build a table of baseline vs mitigated ASR per attack
category (plus overall), confirm benign utility stays high, and chart the before/after.
The goal is a large ASR drop with no utility loss.

In [ ]:
# TODO: comparison table and grouped bar chart of baseline vs mitigated ASR


## Part 7 - Residual risk
Mitigations rarely reach zero. Find at least one attack that still succeeds, explain
why your guardrail missed it, and propose the next control. **Write this up honestly**;
naming residual risk is part of safety work, not a failure. *(double-click to edit)*

## Part 8 - Governance mapping
Turn the engineering into governance.

1. **NIST AI RMF:** map your work to Govern, Map, Measure, Manage (a small table).
2. **Risk register:** for each risk, record impact, likelihood, control, and residual
   risk.
3. **EU AI Act:** in a markdown cell, classify this assistant (unacceptable / high /
   limited / minimal risk) and justify it. Note it is educational, not legal advice,
   and that the Act applies in phases and evolves.

In [ ]:
# TODO: NIST AI RMF mapping table and a risk_register DataFrame


## Part 9 - Model card
Write a short model card: intended use, out-of-scope use, data, safety evaluation
results, mitigations, limitations and oversight, governance. *(double-click to edit)*

## Part 10 - The safety evaluation & governance report
Assemble the one-page report you ship: scope, method, results table (baseline vs
mitigated ASR), mitigations, residual risk, governance (risk register + framework),
and a recommendation. *(double-click to edit)*